# Feature Engineering

Извлекаю признаки из датасета при помощи XGBoost регрессии


In [3]:
import pandas as pd
import numpy as np
import librosa
from pathlib import Path

base_path = Path("..")

split_path = base_path / "data" / "processed" / "data_split"
train_path = split_path / "train.csv"
test_path = split_path / "test.csv"
val_path = split_path / "val.csv"

In [4]:
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
val_df = pd.read_csv(val_path)
train_df.shape, test_df.shape, val_df.shape

((2415, 10), (520, 10), (515, 10))

#### Проверяем на 1

In [8]:
first_path = Path(train_df["wet_path"].iloc[0])
y, sr = librosa.load(first_path, sr=None)
type(y), y.shape, sr

(numpy.ndarray, (240000,), 48000)

In [17]:
def aggregate_feature(feature, f_name):

    f_mean = feature.mean()
    f_std = feature.std()
    f_median = np.median(feature)

    return {
        f"{f_name}_mean": f_mean,
        f"{f_name}_std": f_std,
        f"{f_name}_median": f_median,
    }

In [ ]:
rms = librosa.feature.rms(y=y)

aggregate_feature(rms, "rms")

(np.float32(0.02440065), np.float32(0.03990087), np.float32(0.006486482))

In [20]:
centroid = librosa.feature.spectral_centroid(y=y, sr=sr)

aggregate_feature(centroid, "centroid")

{'centroid_mean': np.float64(1799.3247257307717),
 'centroid_std': np.float64(728.5355114131153),
 'centroid_median': np.float64(1518.8766700121475)}

In [ ]:
bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)

aggregate_feature(bandwidth, "bandwidth")

(1, 469)


(np.float64(2517.4204691062873),
 np.float64(1288.787349766983),
 np.float64(2268.6445590713383))

In [19]:
rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)

aggregate_feature(rolloff, "rolloff")

{'rolloff_mean': np.float64(3022.2381396588485),
 'rolloff_std': np.float64(1843.799653170514),
 'rolloff_median': np.float64(2484.375)}

In [18]:
zrc = librosa.feature.zero_crossing_rate(y=y)

aggregate_feature(zrc, "zcr")

{'zcr_mean': np.float64(0.033993328558102345),
 'zcr_std': np.float64(0.011499821590852515),
 'zcr_median': np.float64(0.033203125)}

In [21]:
features = {}

features.update(aggregate_feature(rms, "rms"))
features.update(aggregate_feature(centroid, "centroid"))
features.update(aggregate_feature(bandwidth, "bandwidth"))
features.update(aggregate_feature(rolloff, "rolloff"))
features.update(aggregate_feature(zrc, "zcr"))

features

{'rms_mean': np.float32(0.02440065),
 'rms_std': np.float32(0.03990087),
 'rms_median': np.float32(0.006486482),
 'centroid_mean': np.float64(1799.3247257307717),
 'centroid_std': np.float64(728.5355114131153),
 'centroid_median': np.float64(1518.8766700121475),
 'bandwidth_mean': np.float64(2517.4204691062873),
 'bandwidth_std': np.float64(1288.787349766983),
 'bandwidth_median': np.float64(2268.6445590713383),
 'rolloff_mean': np.float64(3022.2381396588485),
 'rolloff_std': np.float64(1843.799653170514),
 'rolloff_median': np.float64(2484.375),
 'zcr_mean': np.float64(0.033993328558102345),
 'zcr_std': np.float64(0.011499821590852515),
 'zcr_median': np.float64(0.033203125)}

In [27]:
mfcc = librosa.feature.mfcc(
    y=y,
    sr=sr,
    n_mfcc=13
)
mfcc_features = {}

for i in range (mfcc.shape[0]):
    mfcc_features.update(aggregate_feature(mfcc[i], f"mfcc_{i+1}"))

print(list(mfcc_features.items())[:6])
len(mfcc_features)

[('mfcc_1_mean', np.float32(-483.36322)), ('mfcc_1_std', np.float32(102.59901)), ('mfcc_1_median', np.float32(-493.36777)), ('mfcc_2_mean', np.float32(137.37407)), ('mfcc_2_std', np.float32(62.755207)), ('mfcc_2_median', np.float32(142.5387))]


39

In [28]:
features.update(mfcc_features)

len(features)

54